In [16]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [17]:
import sys
print(sys.executable)

c:\Python313\python.exe


In [18]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\chhap\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\chhap\AppData\Roaming\nltk_data...


True

In [19]:
df_raw = pd.read_csv("dataset/Resume.csv")

print("Shape of raw data:", df_raw.shape)
print("Columns:", df_raw.columns)

df_raw.head()

Shape of raw data: (2484, 4)
Columns: Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='str')


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [20]:
df_clean1 = df_raw[['Resume_str', 'Category']].copy()

print("Shape after selecting useful columns:", df_clean1.shape)

df_clean1.head()

Shape after selecting useful columns: (2484, 2)


,Resume_str,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,HR DIRECTOR Summary Over 2...,HR
3,HR SPECIALIST Summary Dedica...,HR
4,HR MANAGER Skill Highlights ...,HR


In [21]:
df_clean1.isnull().sum()

Resume_str    0
Category      0
dtype: int64

In [23]:
df_clean2 = df_clean1.copy()

# Remove all HTML entities like &nbsp;, &amp;, etc.
df_clean2['text'] = df_clean2['Resume_str'].apply(
    lambda x: re.sub(r'&[a-zA-Z]+;', ' ', x)
)

df_clean2[['Resume_str', 'text']].head(3)

,Resume_str,text
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1,"HR SPECIALIST, US HR OPERATIONS ...","HR SPECIALIST, US HR OPERATIONS ..."
2,HR DIRECTOR Summary Over 2...,HR DIRECTOR Summary Over 2...


In [24]:
df_clean3 = df_clean2.copy()

df_clean3['text'] = df_clean3['text'].str.lower()

df_clean3[['Resume_str','text']].head(3)

,Resume_str,text
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administrator/marketing associate\...
1,"HR SPECIALIST, US HR OPERATIONS ...","hr specialist, us hr operations ..."
2,HR DIRECTOR Summary Over 2...,hr director summary over 2...


In [25]:
df_clean4 = df_clean3.copy()

# Remove any HTML entity like &nbsp; &amp; etc.
df_clean4['text'] = df_clean4['text'].apply(
    lambda x: re.sub(r'&[a-zA-Z]+;', ' ', x)
)

# Remove non-breaking space unicode character if present
df_clean4['text'] = df_clean4['text'].str.replace('\xa0', ' ', regex=False)

df_clean4[['Resume_str','text']].head(3)

,Resume_str,text
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administrator/marketing associate\...
1,"HR SPECIALIST, US HR OPERATIONS ...","hr specialist, us hr operations ..."
2,HR DIRECTOR Summary Over 2...,hr director summary over 2...


In [26]:
df_clean5 = df_clean4.copy()

# Remove emails
df_clean5['text'] = df_clean5['text'].apply(
    lambda x: re.sub(r'\S+@\S+', '', x)
)

df_clean5[['text']].head(3)

,text
0,hr administrator/marketing associate\...
1,"hr specialist, us hr operations ..."
2,hr director summary over 2...


In [27]:
df_clean6 = df_clean5.copy()

# Remove URLs
df_clean6['text'] = df_clean6['text'].apply(
    lambda x: re.sub(r'http\S+|www\S+', '', x)
)

df_clean6[['text']].head(3)

,text
0,hr administrator/marketing associate\...
1,"hr specialist, us hr operations ..."
2,hr director summary over 2...


In [28]:
df_clean7 = df_clean6.copy()

# Remove numbers
df_clean7['text'] = df_clean7['text'].apply(
    lambda x: re.sub(r'\d+', '', x)
)

df_clean7[['text']].head(3)

,text
0,hr administrator/marketing associate\...
1,"hr specialist, us hr operations ..."
2,hr director summary over ...


In [30]:
df_clean8 = df_clean7.copy()

# Replace punctuation with space instead of removing
df_clean8['text'] = df_clean8['text'].apply(
    lambda x: re.sub(r'[^\w\s]', ' ', x)
)

df_clean8[['text']].head(3)

,text
0,hr administrator marketing associate\...
1,hr specialist us hr operations ...
2,hr director summary over ...


In [31]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [32]:
df_clean9 = df_clean8.copy()

df_clean9['text'] = df_clean9['text'].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)

df_clean9[['text']].head(3)

,text
0,hr administrator marketing associate hr admini...
1,hr specialist us hr operations summary versati...
2,hr director summary over years experience in r...


In [33]:
df_clean10 = df_clean9.copy()

df_clean10['text'] = df_clean10['text'].apply(
    lambda x: " ".join([
        word for word in x.split()
        if word not in stop_words and len(word) > 2
    ])
)

df_clean10[['text']].head(3)

,text
0,administrator marketing associate administrato...
1,specialist operations summary versatile media ...
2,director summary years experience recruiting p...


In [34]:
df_clean11 = df_clean10.copy()

df_clean11['text'] = df_clean11['text'].apply(
    lambda x: " ".join([
        lemmatizer.lemmatize(word)
        for word in x.split()
    ])
)

df_clean11[['text']].head(3)

,text
0,administrator marketing associate administrato...
1,specialist operation summary versatile medium ...
2,director summary year experience recruiting pl...


In [35]:
df_final = df_clean11.copy()

df_final = df_final[df_final['text'].str.strip() != ""]

print("Final shape:", df_final.shape)

Final shape: (2483, 3)


In [36]:
print(df_final.columns)

Index(['Resume_str', 'Category', 'text'], dtype='str')


In [37]:
df_final = df_clean11.copy()

df_final = df_final.rename(columns={'text': 'cleaned_resume'})

print(df_final.columns)

Index(['Resume_str', 'Category', 'cleaned_resume'], dtype='str')


In [38]:
df_final = df_final[['Category', 'cleaned_resume']]

print(df_final.shape)
df_final.head()

(2484, 2)


,Category,cleaned_resume
0,HR,administrator marketing associate administrato...
1,HR,specialist operation summary versatile medium ...
2,HR,director summary year experience recruiting pl...
3,HR,specialist summary dedicated driven dynamic ye...
4,HR,manager skill highlight skill department start...
